# Equity Research AI Agent - Interactive Notebook

This notebook provides an interactive interface for the Equity Research AI Agent using LangChain and LangGraph.

## Features
- Process PDF documents (10-K filings, annual reports)
- RAG (Retrieval Augmented Generation) system with ChromaDB
- Yahoo Finance data integration
- Web search capabilities
- Multi-perspective equity research analysis
- Interactive Q&A sessions

## 1. Setup and Imports

In [ ]:
import os
import logging
from pathlib import Path
from dotenv import load_dotenv

from src.rag.pdf_processor import PDFProcessor
from src.rag.vector_store import VectorStoreManager
from src.agent.equity_research_agent import EquityResearchAgent
from src.utils.logging_config import setup_logging

# Load environment variables
load_dotenv()

# Setup logging
setup_logging(level="INFO")
logger = logging.getLogger(__name__)

print("✓ All imports successful!")

## 2. Configuration

Set your configuration parameters here:

In [ ]:
# Configuration
PDF_PATH = "Salesforce, Inc. files (10-K) Basic annual filing, for period end 31-Jan-26 (CRM-US).pdf"
TICKER = "CRM"
PERSIST_DIR = "./chroma_db"
USE_EXISTING_VECTOR_STORE = False  # Set to True to skip PDF processing

# Check for OpenAI API key
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  WARNING: OPENAI_API_KEY not found in environment variables.")
    print("Please set it in a .env file or export it as an environment variable.")
else:
    print("✓ OpenAI API key found!")

## 3. Initialize RAG System

Process PDFs and create/load the vector store:

In [ ]:
def initialize_rag_system(pdf_path: str, use_existing: bool = False) -> VectorStoreManager:
    """
    Initialize the RAG system with PDF documents.
    
    Args:
        pdf_path: Path to PDF file or directory
        use_existing: Whether to load existing vector store
    
    Returns:
        Initialized VectorStoreManager
    """
    vector_store_manager = VectorStoreManager()
    
    # Check if we should load existing vector store
    if use_existing and os.path.exists(PERSIST_DIR):
        logger.info("Loading existing vector store...")
        vector_store_manager.load_vector_store_chroma(PERSIST_DIR)
        print("✓ Loaded existing vector store")
        return vector_store_manager
    
    # Process PDF(s)
    logger.info(f"Processing PDF(s) from {pdf_path}...")
    pdf_processor = PDFProcessor()
    
    if os.path.isfile(pdf_path):
        pdf_data = pdf_processor.process_pdf(pdf_path)
        all_pdf_data = [pdf_data]
    elif os.path.isdir(pdf_path):
        all_pdf_data = pdf_processor.process_directory(pdf_path)
    else:
        raise ValueError(f"Invalid path: {pdf_path}")
    
    # Create documents for vector store
    all_documents = []
    for pdf_data in all_pdf_data:
        documents = vector_store_manager.create_documents_from_pdf_data(pdf_data)
        all_documents.extend(documents)
    
    logger.info(f"Created {len(all_documents)} document chunks from {len(all_pdf_data)} PDF(s)")
    print(f"✓ Created {len(all_documents)} document chunks from {len(all_pdf_data)} PDF(s)")
    
    # Create vector store
    vector_store_manager.create_vector_store_chroma(all_documents, PERSIST_DIR)
    print("✓ Vector store created successfully")
    
    return vector_store_manager

# Initialize the RAG system
vector_store_manager = initialize_rag_system(PDF_PATH, USE_EXISTING_VECTOR_STORE)

## 4. Initialize Agent

Create the Equity Research Agent:

In [ ]:
# Initialize agent
logger.info("Initializing equity research agent...")
agent = EquityResearchAgent(vector_store_manager)
print("✓ Equity Research Agent initialized successfully!")

## 5. Run Analysis

### Option A: Single Question Analysis

Ask a specific question about the company:

In [ ]:
# Define your question here
question = "Provide a comprehensive equity research summary analyzing the company's financial performance, business model, risks, and growth opportunities."

# Run analysis
logger.info(f"Running analysis for: {question}")
result = agent.run(question, ticker=TICKER)

# Display results
print("\n" + "="*80)
print("EQUITY RESEARCH ANALYSIS")
print("="*80)
print(f"\nQuestion: {result['question']}")
if result['ticker']:
    print(f"Ticker: {result['ticker']}")
print(f"\n{result['answer']}")
print("\n" + "="*80)

### Option B: Custom Question

Ask your own custom question:

In [ ]:
# Ask a custom question
custom_question = "What are Salesforce's main business segments and revenue streams?"

result = agent.run(custom_question, ticker=TICKER)

print("\n" + "="*80)
print("ANALYSIS RESULT")
print("="*80)
print(f"\nQuestion: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print("\n" + "="*80)

### Option C: Interactive Q&A Session

Ask multiple questions in sequence. Each question can build on previous context:

In [ ]:
# Define your questions
questions = [
    "What were Salesforce's total revenues in the most recent fiscal year?",
    "How does this compare to the previous year?",
    "What are the key drivers of revenue growth?"
]

print("\n" + "="*80)
print("INTERACTIVE Q&A SESSION")
print("="*80 + "\n")

for i, q in enumerate(questions, 1):
    print(f"Question {i}: {q}")
    result = agent.run(q, ticker=TICKER)
    print(f"\nAnswer:\n{result['answer']}")
    print("\n" + "-"*80 + "\n")

## 6. Alternative: Manual Interactive Mode

Run this cell to enter a fully interactive mode (similar to the CLI --interactive flag):

In [ ]:
def run_interactive_mode(agent: EquityResearchAgent, ticker: str = None):
    """
    Run the agent in interactive Q&A mode.
    
    Note: In Jupyter, you can simply run cells multiple times instead.
    This function is provided for completeness.
    """
    print("\n" + "="*80)
    print("INTERACTIVE EQUITY RESEARCH AGENT")
    print("="*80)
    print("\nAsk questions about the company and financial data.")
    print("Type 'exit' or 'quit' to end the session.\n")
    
    while True:
        try:
            question = input("Your question: ").strip()
            
            if question.lower() in ['exit', 'quit', 'q']:
                print("Ending session. Goodbye!")
                break
            
            if not question:
                continue
            
            result = agent.run(question, ticker=ticker)
            
            print(f"\n{result['answer']}\n")
            print("-" * 80 + "\n")
            
        except KeyboardInterrupt:
            print("\n\nSession interrupted. Goodbye!")
            break
        except Exception as e:
            logger.error(f"Error during interactive session: {e}")
            print(f"\nError: {e}\n")

# Uncomment to run interactive mode
# run_interactive_mode(agent, ticker=TICKER)

## 7. View Conversation History (Optional)

View the agent's internal state and conversation history:

In [ ]:
# This would show the conversation history if you want to inspect it
# Note: The actual implementation depends on the agent's state management
print("To view the agent's conversation history, you can access agent.state['conversation_history']")
print("This is useful for debugging and understanding the agent's reasoning process.")

## 8. Quick Reference

### Common Questions to Try:

1. **Financial Performance:**
   - "What were the total revenues in the most recent fiscal year?"
   - "What is the company's profitability trend?"
   - "Analyze the cash flow situation"

2. **Business Analysis:**
   - "What are the main business segments?"
   - "Who are the key competitors?"
   - "What is the company's competitive advantage?"

3. **Risk Assessment:**
   - "What are the main business risks?"
   - "What regulatory risks does the company face?"
   - "Analyze market and industry risks"

4. **Growth Analysis:**
   - "What are the growth opportunities?"
   - "What is driving revenue growth?"
   - "Analyze the company's growth strategy"

5. **Valuation:**
   - "What is the current P/E ratio?"
   - "How is the company valued compared to peers?"
   - "Provide a valuation analysis"

### Tips:
- The agent automatically fetches Yahoo Finance data when you provide a ticker
- Financial data is added to the RAG system for retrieval across queries
- You can ask follow-up questions that reference previous answers
- The agent combines information from PDFs, Yahoo Finance, and web search